In [ ]:
import sys
print(sys.executable)
import optuna
print(optuna.__version__)


In [ ]:
# Jalankan ini di notebook kamu:
import sys
!{sys.executable} -m pip install "numpy<2.0" --force-reinstall


In [ ]:
import torch
print(torch.__version__)


In [4]:
pip install ultralytics --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.8 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.46
    Uninstalling ultralytics-8.4.46:
      Successfully uninstalled ultralytics-8.4.46
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import optuna
from ultralytics import YOLO
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
%matplotlib inline

In [ ]:
# base cctv dataset
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fh3FKMZkgt1ugTO9Hjvz")
project = rf.workspace("sanka").project("cctv-xb1jx")
version = project.version(2)
dataset = version.download("yolov11")

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="CToH9bk7hkApGXIzLFMf")
project = rf.workspace("benediktas-workspace").project("person-tracking")
version = project.version(2)
dataset = version.download("yolo26")
                

                

In [ ]:
# data cctv tambahan lagi, dari hans

from roboflow import Roboflow

rf = Roboflow(api_key="fh3FKMZkgt1ugTO9Hjvz")
project = rf.workspace("floor-plan-recognition-research").project("scut-head-usw52")
version = project.version(1)
dataset = version.download("yolo26")

In [2]:
#initiate 

class CFG:
    # General Config
    SEED = 123
    DEVICE = 'cpu'   # ganti dari mps → cpu, MPS masih banyak bug untuk YOLO training
    DEBUG = False

    # Dataset
    DATA_YAML = "SCUT-HEAD-1/data.yaml"

    # Model & Training Config
    BASE_MODEL = "yolo26n.pt"
    EPOCHS = 3 if DEBUG else 50
    BATCH_SIZE = 8
    IMG_SIZE = 416   # turunkan dari 640 → 416 untuk kompensasi speed di CPU

    OPTIMIZER = "SGD"
    LR0 = 1e-3
    LR_FACTOR = 0.01
    MOMENTUM = 0.9
    WEIGHT_DECAY = 5e-4
    WARMUP_EPOCHS = 5
    DROPOUT = 0.0

    # Scheduler
    SCHEDULER = "cosine"

    # Early Stopping
    PATIENCE = 20

    # Augmentation
    CLOSE_MOSAIC = 10

    # Memory
    WORKERS = 0

    # Tracking
    IOU = 0.45
    CONF = 0.001
    max_det = 200


In [3]:
model = YOLO(CFG.BASE_MODEL).to(CFG.DEVICE)

results = model.train(
    data=CFG.DATA_YAML,
    epochs=CFG.EPOCHS,
    batch=CFG.BATCH_SIZE,
    imgsz=CFG.IMG_SIZE,
    optimizer=CFG.OPTIMIZER,
    lr0=CFG.LR0,
    lrf=CFG.LR_FACTOR,
    weight_decay=CFG.WEIGHT_DECAY,
    warmup_epochs=CFG.WARMUP_EPOCHS,
    cos_lr=True if CFG.SCHEDULER == "cosine" else False,
    patience=CFG.PATIENCE,
    close_mosaic=CFG.CLOSE_MOSAIC,
    workers=CFG.WORKERS,
    cache=False,
    amp=False,   # matikan mixed precision — fix shape mismatch di MPS
    verbose=True,
    iou=CFG.IOU,
    conf=CFG.CONF,
    max_det=CFG.max_det,
    device=CFG.DEVICE,
    plots=False,
)


New https://pypi.org/project/ultralytics/8.4.47 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.46 🚀 Python-3.12.2 torch-2.11.0 CPU (Apple M2)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SCUT-HEAD-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.45, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=200, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-28, nbs=64, nms=False,

KeyboardInterrupt: 